# Convert OAM-TCD to shards

Run on a CPU runtime (about 2 hours). Set `REPO_SHA` in the next cell, then run the cells in order.

Reads `restor/tcd` (CC-BY-4.0) and `restor/tcd-nc` (CC-BY-NC-4.0) from Hugging Face at pinned revisions; `restor/tcd-sa` (CC-BY-SA) is never read. Each 2048 px, 10 cm tile becomes one region with z19 and z20 chips: tree boxes are labels, canopy (tree groups, minus the trees inside) and trees over 30 m are masked. Validation fold 0 is `val`, folds 1-4 `train`, the Hugging Face test split `test` (benchmark only). Tiles within 1 km of HOT's Freetown tiles are dropped.

The parquet files are streamed one record batch at a time and only the finished shards are written to Drive. The last cells write the OAM-TCD test benchmark part (`benchmark_part_oamtcd.json`) to freeze in the repo, and a contact sheet.

The repo is private: add a read-only GitHub token (Contents: read) under Colab Secrets (the key icon) as `GITHUB_TOKEN`, with notebook access on.

In [ ]:
SOURCE = "oamtcd"
REPO_SHA = ""
REVISION_TCD = "d97d4da0ebbb6e249ae95ac5e19656babd972eb2"  # restor/tcd, checked 2026-10-29
REVISION_TCD_NC = "7c2c9b42e5e71a45eee7657842861db5836f8778"  # restor/tcd-nc, checked 2026-10-29
RESOLVE_ZIP = "raw/resolve/Ecoregions2017.zip"  # relative to the Drive root

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/fair_trees")

assert REPO_SHA, "set REPO_SHA to the pushed commit"
# the repo is private: a read-only GitHub token in Colab Secrets (key icon), named GITHUB_TOKEN
REPO = "github.com/OmdenaAI/HOTOSM-fAIr-Trees.git"
token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/fair_trees
# git clone through subprocess so the token never reaches the cell output
url = f"https://x-access-token:{token}@{REPO}"
clone = subprocess.run(["git", "clone", "-q", url, "/content/fair_trees"])
# checked by hand: a CalledProcessError message would print the URL, token included
if clone.returncode:
    raise RuntimeError(f"git clone failed (exit {clone.returncode}): check the GITHUB_TOKEN secret")
# drop the token from the saved remote so it is not left in .git/config
git = ["git", "-C", "/content/fair_trees"]
subprocess.run([*git, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
subprocess.run([*git, "checkout", "-q", REPO_SHA], check=True)
del token, url

%pip install /content/fair_trees -r /content/fair_trees/experiments/requirements.txt
sys.path.insert(0, "/content/fair_trees")

In [ ]:
import json
import shutil
import zipfile

from huggingface_hub import hf_hub_download

from experiments.label_study.hot import REPO_ID, REVISION, read_hot

RAW = Path("/content/oamtcd_raw")
FILES = {
    "tcd": (
        "restor/tcd",
        REVISION_TCD,
        {
            "train": [f"data/train-0000{i}-of-00007.parquet" for i in range(7)],
            "test": ["data/test-00000-of-00001.parquet"],
        },
    ),
    "tcd-nc": (
        "restor/tcd-nc",
        REVISION_TCD_NC,
        {
            "train": ["data/train-00000-of-00001.parquet"],
            "test": ["data/test-00000-of-00001.parquet"],
        },
    ),
}
parquet = {
    dataset: {
        split: [
            Path(hf_hub_download(repo, f, repo_type="dataset", revision=rev, local_dir=RAW / dataset))
            for f in files
        ]
        for split, files in splits.items()
    }
    for dataset, (repo, rev, splits) in FILES.items()
}
print({d: {s: len(p) for s, p in v.items()} for d, v in parquet.items()})

# HOT's tiles, so OAM-TCD tiles near Freetown are kept out of training
hot_files = {
    name: hf_hub_download(
        REPO_ID,
        f"stac/project-50661/{name}.parquet",
        repo_type="dataset",
        revision=REVISION,
        local_dir="/content/hot_raw",
    )
    for name in ("labels", "tiles")
}
hot_bbox = tuple(
    float(v) for v in read_hot(hot_files["labels"], hot_files["tiles"]).tiles.to_crs(4326).total_bounds
)
print("HOT tiles bbox (west, south, east, north):", hot_bbox)

resolve_dir = Path("/content/resolve")
resolve_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DRIVE / RESOLVE_ZIP) as z:
    z.extractall(resolve_dir)

In [ ]:
import pyarrow.parquet as pq

from experiments.converters.oamtcd import category_ids

# which COCO category is the tree and which the canopy: inferred from the data, checked, printed
first = next(
    pq.ParquetFile(parquet["tcd"]["train"][0]).iter_batches(columns=["coco_annotations"], batch_size=300)
)
ids = category_ids(json.loads(a or "[]") for a in first.column("coco_annotations").to_pylist())
print("categories:", ids)

In [ ]:
import logging

from experiments.biome import resolve_biome_fn
from experiments.converters.oamtcd import convert

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
COLUMNS = ["image_id", "image", "width", "height", "bounds", "crs", "validation_fold", "coco_annotations"]


def rows():
    """Yield one dict per tile, one record batch at a time, tagged with its dataset and Hugging Face split."""
    for dataset, splits in parquet.items():
        for hf_split, paths in splits.items():
            for path in paths:
                for batch in pq.ParquetFile(path).iter_batches(columns=COLUMNS, batch_size=16):
                    for r in batch.to_pylist():
                        yield {**r, "image": r["image"]["bytes"], "dataset": dataset, "hf_split": hf_split}


converted = Path("/content/converted") / SOURCE
# scratch output on the Colab VM only; convert refuses a non-empty output directory
shutil.rmtree(converted, ignore_errors=True)
shutil.rmtree("/content/oamtcd-work", ignore_errors=True)
stats = convert(
    rows(),
    converted,
    ids=ids,
    hot_bbox_4326=hot_bbox,
    biome_fn=resolve_biome_fn(next(resolve_dir.rglob("*.shp"))),
    work_dir=Path("/content/oamtcd-work"),
)
print(json.dumps(stats, indent=2))

In [ ]:
import datetime

from experiments.benchmark import regions_part
from experiments.shards import manifest_hash, write_shards

shard_dir = DRIVE / "shards" / SOURCE
if (shard_dir / "manifest.json").exists():
    # keep the previous conversion (the runs trained on it record its manifest) instead of writing over it
    old = shard_dir.with_name(f"{SOURCE}-{manifest_hash(shard_dir / 'manifest.json')[:8]}")
    shard_dir.rename(old)
    print("previous shards moved to", old)
write_shards(converted, shard_dir, source=SOURCE, converter_sha=REPO_SHA)
print("manifest sha256:", manifest_hash(shard_dir / "manifest.json"))
shutil.copyfile(converted / "stats.json", shard_dir / "stats.json")

# the OAM-TCD test part of the benchmark: send this file to freeze it in the repo
part = regions_part(
    converted.parent,
    SOURCE,
    split="test",
    zooms=[20, 19],
    window_m=40.0,
    frozen_on=datetime.date.today().isoformat(),
)
(shard_dir / "benchmark_part_oamtcd.json").write_text(json.dumps(part, indent=2) + "\n")
print(len(part["unit_ids"]), "test units; written to", shard_dir / "benchmark_part_oamtcd.json")

In [ ]:
import random

from affine import Affine
from IPython.display import Image, display
from PIL import Image as PILImage
from PIL import ImageDraw

from fair_trees.dataset import labels_to_boxes, load_geojson
from fair_trees.georef import tile_bounds_3857
from fair_trees.preprocess import CHIP_SUFFIXES, read_chip

THUMB = 256
train_dirs = [
    p.parent for p in converted.glob("*/region.json") if json.loads(p.read_text())["split"] == "train"
]
chip_paths = sorted(p for d in train_dirs for p in d.glob("z20/chips/*") if p.suffix.lower() in CHIP_SUFFIXES)
picked = random.Random(0).sample(chip_paths, min(16, len(chip_paths)))
contact = PILImage.new("RGB", (4 * THUMB, 4 * THUMB))
for i, path in enumerate(picked):
    rgb, _, (x, y, z) = read_chip(path, use_sidecar=False)
    minx, miny, maxx, maxy = tile_bounds_3857(x, y, z)
    res = (maxx - minx) / rgb.shape[1]
    transform = Affine(res, 0, minx, 0, -res, maxy)
    region = path.parents[2]
    tile = PILImage.fromarray(rgb[..., :3])
    draw = ImageDraw.Draw(tile)
    ignore = load_geojson(region / "ignore.geojson")["features"]
    # trees red, canopy grey, trees over 30 m orange
    layers = (
        ([f for f in ignore if f["properties"]["reason"] == "canopy"], (128, 128, 128)),
        ([f for f in ignore if f["properties"]["reason"] == "large_tree"], (255, 140, 0)),
        (load_geojson(region / "labels.geojson")["features"], (255, 0, 0)),
    )
    for features, colour in layers:
        for x0, y0, x1, y1 in labels_to_boxes({"features": features}, transform):
            draw.rectangle((x0, y0, x1, y1), outline=colour, width=2)
    contact.paste(tile.resize((THUMB, THUMB)), ((i % 4) * THUMB, (i // 4) * THUMB))
sheet = shard_dir / "qa-contact-sheet.png"
contact.save(sheet)
display(Image(filename=str(sheet)))

In [ ]:
import time

largest = max((p for p in shard_dir.iterdir() if p.suffix == ".tar"), key=lambda p: p.stat().st_size)
start = time.monotonic()
shutil.copyfile(largest, Path("/content") / largest.name)
elapsed = time.monotonic() - start
print(f"{largest.name}: {largest.stat().st_size / 1e6 / elapsed:.1f} MB/s")
# Record this in experiments/DATA_SOURCES.md; restaging must take under 10 minutes.

In [ ]:
drive.flush_and_unmount()